In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, DoubleType, TimestampType
)

BRONZE_TABLE = "nyc_taxi_datalake.bronze.yellow_taxi"
ZONE_TABLE   = "nyc_taxi_datalake.bronze.taxi_zones"
SILVER_TABLE = "nyc_taxi_datalake.silver.yellow_taxi"

YEARS_MONTHS = [(2024, m) for m in range(1, 13)]

## Idempotency

In [0]:
def already_processed(year: int, month: int) -> bool:
    try:
        count = spark.table(SILVER_TABLE) \
                     .filter(
                         (F.col("_pickup_year")  == year) &
                         (F.col("_pickup_month") == month)
                     ).limit(1).count()
        return count > 0
    except Exception:
        return False

## Reading From The Bronze Layer + Cast Types

In [0]:
def read_and_cast(year: int, month: int):
    df = spark.table(BRONZE_TABLE).filter(
                  (F.col("_pickup_year")  == year) &
                  (F.col("_pickup_month") == month)
    )
    print(f"Read from Bronze: {df.count():,} rows")

    # Explicitly cast every column to its correct type

    df = df \
        .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType())) \
        .withColumn("RatecodeID",      F.col("RatecodeID").cast(IntegerType())) \
        .withColumn("payment_type",    F.col("payment_type").cast(IntegerType()))

    # Rename columns to the standard snake_case for consistency
    df = df \
        .withColumnRenamed("VendorID",             "vendor_id") \
        .withColumnRenamed("RatecodeID",            "rate_code_id") \
        .withColumnRenamed("PULocationID",          "pu_location_id") \
        .withColumnRenamed("DOLocationID",          "do_location_id") \
        .withColumnRenamed("Airport_fee",           "airport_fee")

    return df

## Filter Invalid Rows

In [0]:
def filter_invalid(df):

    count_before = df.count()

    # calculate duration
    df = df.withColumn(
        "_duration_check",
        (F.unix_timestamp("tpep_dropoff_datetime") -
         F.unix_timestamp("tpep_pickup_datetime")) / 60
    )

    df = df.filter(
        # Distance
        (F.col("trip_distance")   >  0)    &
        (F.col("trip_distance")   <  100)  &

        (F.col("fare_amount")     >  0)    &
        (F.col("fare_amount")     <  500)  &

        (F.col("passenger_count") >= 1)    &
        (F.col("passenger_count") <= 6)    &

        # Timestamps must exist
        (F.col("tpep_pickup_datetime").isNotNull())  &
        (F.col("tpep_dropoff_datetime").isNotNull()) &

        # Dropoff must be after pickup
        (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")) &

        # Duration between 1 min and 3 hours
        (F.col("_duration_check") >= 1)    &
        (F.col("_duration_check") <= 180)  &

        # RateCode
        (F.col("rate_code_id").isin([1, 2, 3, 4, 5, 6]))
    )

    # Drop the temporary column used only for filtering
    df = df.drop("_duration_check")

    count_after = df.count()
    dropped     = count_before - count_after
    dropped_pct = (dropped / count_before) * 100

    print(f" Before : {count_before:,}")
    print(f" After  : {count_after:,}")
    print(f"  Dropped : {dropped:,} ({dropped_pct:.2f}%)")

    return df

## Derive New Columns

In [0]:
def derive_columns(df):

    df = df \
        .withColumn(
            "trip_duration_min",
            F.round(
                (F.unix_timestamp("tpep_dropoff_datetime") -
                 F.unix_timestamp("tpep_pickup_datetime")) / 60,
                2
            )
        ) \
        .withColumn(
            "pickup_hour", F.hour("tpep_pickup_datetime")
        ) \
        .withColumn(
            "pickup_dow", F.dayofweek("tpep_pickup_datetime")
        ) \
        .withColumn(
            # 
            "pickup_date", F.to_date("tpep_pickup_datetime")
        ) \
        .withColumn(
            "speed_mph",
            F.round(
                F.col("trip_distance") / (F.col("trip_duration_min") / 60),
                2
            )
        ) \
        .withColumn(
            "tip_pct",
            F.round(
                F.col("tip_amount") / F.nullif(F.col("fare_amount"), F.lit(0)) * 100,
                2
            )
        ) \
        .withColumn(
            # Total surcharges in one column
            "total_surcharges",
            F.round(
                F.col("extra") +
                F.col("mta_tax") +
                F.col("improvement_surcharge") +
                F.coalesce(F.col("congestion_surcharge"), F.lit(0.0)) +
                F.coalesce(F.col("airport_fee"), F.lit(0.0)),
                2
            )
        )

    # Filter impossible speeds after deriving speed
    df = df.filter(F.col("speed_mph") < 120)

    print("Derived: trip_duration_min, pickup_hour, pickup_dow, pickup_date, speed_mph, tip_pct, total_surcharges")
    return df

## Joining Zones Table For Enrichment

In [0]:
def join_zones(df):

    zones = spark.table(ZONE_TABLE).select(
        F.col("LocationID"),
        F.col("Borough"),
        F.col("Zone")
    )

    df = df.join(
        # join for pickup location
        zones.select(
            F.col("LocationID").alias("_pu_id"),
            F.col("Zone").alias("pickup_zone"),
            F.col("Borough").alias("pickup_borough")
        ),
        df["pu_location_id"] == F.col("_pu_id"),
        #left join to keep all the trips even if they don't have a zone
        how="left"
    ).drop("_pu_id")

    # Join for dropoff location
    df = df.join(
        zones.select(
            F.col("LocationID").alias("_do_id"),
            F.col("Zone").alias("dropoff_zone"),
            F.col("Borough").alias("dropoff_borough")
        ),
        df["do_location_id"] == F.col("_do_id"),
        how="left"
    ).drop("_do_id")

    print("Joined: pickup_zone, pickup_borough, dropoff_zone, dropoff_borough")
    return df

In [0]:
def write_to_silver(df, year: int, month: int):

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("partitionOverwriteMode", "dynamic") \
        .partitionBy("_pickup_year", "_pickup_month") \
        .option("mergeSchema", "true") \
        .saveAsTable(SILVER_TABLE)

    print(f"Written to Silver: {SILVER_TABLE} — partition {year}-{month:02d}")

## running the full pipeline

In [0]:
for year, month in YEARS_MONTHS:
    print(f"\n{'='*40}")
    print(f"Processing Silver: {year}-{month:02d}")
    print(f"{'='*40}")

    if already_processed(year, month):
        print(f"Already in Silver, skipping {year}-{month:02d}")
        continue

    df = read_and_cast(year, month)
    df = filter_invalid(df)
    df = derive_columns(df)
    df = join_zones(df)
    write_to_silver(df, year, month)

print("\n Silver processing complete")

#Verify Silver


In [0]:
df_silver = spark.table(SILVER_TABLE)
df_bronze = spark.table(BRONZE_TABLE)

bronze_count = df_bronze.count()
silver_count = df_silver.count()
dropped_pct  = ((bronze_count - silver_count) / bronze_count) * 100

print(f"Bronze rows : {bronze_count:,}")
print(f"Silver rows : {silver_count:,}")
print(f"Dropped     : {bronze_count - silver_count:,} ({dropped_pct:.2f}%)")

print("\nQuick sanity checks on value ranges:")

df_silver.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("trip_duration_min").alias("min_duration"),
    F.max("trip_duration_min").alias("max_duration"),
    F.min("speed_mph").alias("min_speed"),
    F.max("speed_mph").alias("max_speed"),
    F.min("passenger_count").alias("min_pax"),
    F.max("passenger_count").alias("max_pax")
).show()

print("\nJust a quick look at the enriched columns:")

df_silver.select(
    "tpep_pickup_datetime",
    "trip_duration_min",
    "pickup_hour",
    "pickup_dow",
    "speed_mph",
    "tip_pct",
    "total_surcharges",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "dropoff_borough"
).show(5, truncate=False)